In [1]:
# Step 1: Locate source files
import os
import pandas as pd

for dirpath, dirnames, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirpath, filename))

/kaggle/input/datasets/asaniczka/1-3m-linkedin-jobs-and-skills-2024/job_summary.csv
/kaggle/input/datasets/asaniczka/1-3m-linkedin-jobs-and-skills-2024/job_skills.csv
/kaggle/input/datasets/asaniczka/1-3m-linkedin-jobs-and-skills-2024/linkedin_job_postings.csv


In [2]:
# Step 2: Build exact match skill population (business analysis / business analyst)
skills_path = '/kaggle/input/datasets/asaniczka/1-3m-linkedin-jobs-and-skills-2024/job_skills.csv'

target_skills = {'business analysis', 'business analyst'}
skill_match_job_links = set()

for chunk in pd.read_csv(skills_path, usecols=['job_link', 'job_skills'], chunksize=100_000):
    for job_link, skills_str in zip(chunk['job_link'], chunk['job_skills'].fillna('')):
        individual_skills = [s.strip().lower() for s in skills_str.split(',')]
        if target_skills.intersection(individual_skills):
            skill_match_job_links.add(job_link)

print(f"Exact-match skill population (all countries): {len(skill_match_job_links)}")

Exact-match skill population (all countries): 7602


In [3]:
# Step 3: Identify explicit "Business Analyst" titles missed by the skill rule
postings_path = '/kaggle/input/datasets/asaniczka/1-3m-linkedin-jobs-and-skills-2024/linkedin_job_postings.csv'

cols_needed = ['job_link', 'job_title', 'company', 'job_location', 'first_seen', 'job_level', 'job_type', 'search_country']

ba_title_job_links = set()
total_rows_seen = 0

for chunk in pd.read_csv(postings_path, usecols=cols_needed, chunksize=100_000):
    total_rows_seen += len(chunk)
    mask = chunk['job_title'].str.contains('business analyst', case=False, na=False)
    ba_title_job_links.update(chunk.loc[mask, 'job_link'])

print(f"Total rows scanned: {total_rows_seen}")
print(f"Postings with 'Business Analyst' in title: {len(ba_title_job_links)}")

# Checking overlap with our skill-match set
already_captured = ba_title_job_links.intersection(skill_match_job_links)
missed_by_skill_rule = ba_title_job_links - skill_match_job_links

print(f"\nAlready captured by skill rule: {len(already_captured)}")
print(f"Missed by skill rule (new): {len(missed_by_skill_rule)}")

Total rows scanned: 1348454
Postings with 'Business Analyst' in title: 4576

Already captured by skill rule: 2450
Missed by skill rule (new): 2126


In [4]:
# Step 4: Validate and add Business System(s) Analyst / Business Process Analyst title families
captured_so_far = skill_match_job_links.union(ba_title_job_links)

business_system_analyst_job_links = set()
business_process_analyst_job_links = set()

for chunk in pd.read_csv(postings_path, usecols=cols_needed, chunksize=100_000):
    titles_lower = chunk['job_title'].str.lower().fillna('')
    
    # Business System(s) Analyst
    bsys_mask = titles_lower.str.contains('business system analyst') | titles_lower.str.contains('business systems analyst')
    bsys_new = chunk.loc[bsys_mask & ~chunk['job_link'].isin(captured_so_far), 'job_link']
    business_system_analyst_job_links.update(bsys_new)
    
    # Business Process Analyst
    bproc_mask = titles_lower.str.contains('business process analyst')
    bproc_new = chunk.loc[bproc_mask & ~chunk['job_link'].isin(captured_so_far), 'job_link']
    business_process_analyst_job_links.update(bproc_new)

print(f"Business System(s) Analyst - new (not already captured): {len(business_system_analyst_job_links)}")
print(f"Business Process Analyst - new (not already captured): {len(business_process_analyst_job_links)}")

Business System(s) Analyst - new (not already captured): 672
Business Process Analyst - new (not already captured): 82


In [5]:
# Step 5: Combine refined population and filter to United States search records
refined_population_all_countries = skill_match_job_links.union(ba_title_job_links).union(
    business_system_analyst_job_links).union(business_process_analyst_job_links)

print(f"Refined BA population (all countries): {len(refined_population_all_countries)}")

# Filter to US search records
refined_us_job_links = set()
total_rows_seen = 0

for chunk in pd.read_csv(postings_path, usecols=cols_needed, chunksize=100_000):
    total_rows_seen += len(chunk)
    mask = chunk['job_link'].isin(refined_population_all_countries) & (chunk['search_country'] == 'United States')
    refined_us_job_links.update(chunk.loc[mask, 'job_link'])

print(f"\nRefined BA population (US only): {len(refined_us_job_links)}")

Refined BA population (all countries): 10482

Refined BA population (US only): 8610


In [6]:
# Step 6: Pull full posting details and skills, merge on job_link (LEFT join)
matched_postings_list = []
total_rows_seen = 0

for chunk in pd.read_csv(postings_path, usecols=cols_needed, chunksize=100_000):
    total_rows_seen += len(chunk)
    mask = chunk['job_link'].isin(refined_us_job_links)
    matched_postings_list.append(chunk.loc[mask])

postings_filtered2 = pd.concat(matched_postings_list, ignore_index=True)
print(f"Postings matched: {len(postings_filtered2)}")

# Pull in skills for these postings
skills_filtered_list = []
for chunk in pd.read_csv(skills_path, usecols=['job_link', 'job_skills'], chunksize=100_000):
    mask = chunk['job_link'].isin(refined_us_job_links)
    skills_filtered_list.append(chunk.loc[mask])

skills_filtered2 = pd.concat(skills_filtered_list, ignore_index=True)
print(f"Skills rows matched: {len(skills_filtered2)}")

# Merge, LEFT join, drop search_country
combined2 = postings_filtered2.merge(skills_filtered2, on='job_link', how='left')
combined2 = combined2.drop(columns=['search_country'])

print(f"\nCombined rows: {len(combined2)}")
print(f"Postings with no skills match after join: {combined2['job_skills'].isna().sum()}")

combined2.head()

Postings matched: 8610
Skills rows matched: 8487

Combined rows: 8610
Postings with no skills match after join: 124


,job_link,job_title,company,job_location,first_seen,job_level,job_type,job_skills
0,https://www.linkedin.com/jobs/view/senior-proc...,"Senior Process Innovation Engineer, Amazon Log...",Amazon,"Bellevue, WA",2024-01-12,Mid senior,Onsite,"CAD, MS Excel, MS Project, Automation, Softwar..."
1,https://www.linkedin.com/jobs/view/secret-clea...,Secret cleared Business Analyst,Latitude Inc,"Crystal City, VA",2024-01-13,Mid senior,Onsite,"Microsoft Office 365, Microsoft Teams, SharePo..."
2,https://www.linkedin.com/jobs/view/business-an...,Business Analyst (11076-1),RAPS Consulting Inc,"Columbia, SC",2024-01-14,Mid senior,Onsite,"Technical Writing, Information Security, Micro..."
3,https://www.linkedin.com/jobs/view/payment-swi...,Payment (Swift) (Business Analyst),Resource Consultings Services Inc,"Lake Mary, FL",2024-01-16,Mid senior,Onsite,NaN
4,https://www.linkedin.com/jobs/view/principal-b...,Principal Business Analyst,NextEra Energy Services,"Riviera Beach, FL",2024-01-14,Mid senior,Onsite,"NEER, FPL, Reliability reporting, GADS applica..."


In [7]:
# Step 7: Save checkpoint. Run duplicate, location, encoding and job_level checks
combined2.to_csv('/kaggle/working/ba_refined_checkpoint.csv', index=False)
print(f"Checkpoint saved: {combined2.shape}")

# Duplicate checks
dup_job_link = combined2['job_link'].duplicated().sum()
print(f"\n1. Exact duplicate job_links: {dup_job_link}")

combined2['is_likely_repost'] = combined2.duplicated(subset=['job_title', 'company', 'job_location'], keep=False)
print(f"2. Likely reposts flagged: {combined2['is_likely_repost'].sum()}")

# job_location completeness
missing_location = combined2['job_location'].isna().sum()
print(f"3. Missing job_location: {missing_location} ({100*missing_location/len(combined2):.2f}%)")
print(f"   Distinct job_location values: {combined2['job_location'].nunique()}")

# Text encoding
nbsp_count = combined2['job_title'].str.contains('\xa0', na=False).sum()
bad_dash_count = combined2['job_title'].str.contains('â€', na=False).sum()
print(f"4. Titles with non-breaking space: {nbsp_count}")
print(f"   Titles with mis-encoded dash: {bad_dash_count}")

# first_seen range
combined2['first_seen'] = pd.to_datetime(combined2['first_seen'])
print(f"5. first_seen range: {combined2['first_seen'].min()} to {combined2['first_seen'].max()}")

# job_level
print(f"\n6. job_level values:")
print(combined2['job_level'].value_counts(dropna=False))

Checkpoint saved: (8610, 8)

1. Exact duplicate job_links: 0
2. Likely reposts flagged: 724
3. Missing job_location: 0 (0.00%)
   Distinct job_location values: 1564
4. Titles with non-breaking space: 20
   Titles with mis-encoded dash: 0
5. first_seen range: 2024-01-12 00:00:00 to 2024-01-17 00:00:00

6. job_level values:
job_level
Mid senior    7510
Associate     1100
Name: count, dtype: int64


In [8]:
# Step 8: Fix non-breaking-space characters in job_title
combined2['job_title'] = combined2['job_title'].str.replace('\xa0', ' ', regex=False)
still_has_nbsp = combined2['job_title'].str.contains('\xa0', na=False).sum()
print(f"Titles still containing non-breaking space: {still_has_nbsp}")

Titles still containing non-breaking space: 0


In [9]:
# Step 9: Build sensitivity population (title + company + location)
sensitivity_population = combined2.drop_duplicates(subset=['job_title', 'company', 'job_location'], keep='first')

print(f"Original refined U.S. BA population: {len(combined2)}")
print(f"Sensitivity population (deduplicated): {len(sensitivity_population)}")

excluded_count = len(combined2) - len(sensitivity_population)
excluded_pct = 100 * excluded_count / len(combined2)
print(f"Excluded for sensitivity test: {excluded_count} ({excluded_pct:.2f}%)")

Original refined U.S. BA population: 8610
Sensitivity population (deduplicated): 8181
Excluded for sensitivity test: 429 (4.98%)


In [10]:
# Step 10: Check job_title fragmentation and apply light text normalisation
distinct_titles = combined2['job_title'].nunique()
title_counts = combined2['job_title'].value_counts()
titles_occurring_once = (title_counts == 1).sum()

print(f"Total postings: {len(combined2)}")
print(f"Distinct job titles: {distinct_titles}")
print(f"Titles occurring only once: {titles_occurring_once}")

# Light normalisation
import re

combined2['job_title_normalized'] = combined2['job_title'].str.lower()
combined2['job_title_normalized'] = combined2['job_title_normalized'].str.replace(r'[.,]', '', regex=True)

replacements = {
    r'\bsr\b': 'senior',
    r'\bjr\b': 'junior',
    r'\bsystems?\b': 'system',
}
for pattern, replacement in replacements.items():
    combined2['job_title_normalized'] = combined2['job_title_normalized'].str.replace(pattern, replacement, regex=True)

combined2['job_title_normalized'] = combined2['job_title_normalized'].str.strip().str.replace(r'\s+', ' ', regex=True)

print(f"\nDistinct titles after light normalization: {combined2['job_title_normalized'].nunique()}")

print(f"\nTop 20 most frequent job titles:")
print(combined2['job_title'].value_counts().head(20))

Total postings: 8610
Distinct job titles: 4989
Titles occurring only once: 4252

Distinct titles after light normalization: 4867

Top 20 most frequent job titles:
job_title
Business Analyst                                            768
JDE Business Analyst (Supply Chain)                         163
JD Edwards Business Analyst                                 150
Business Systems Analyst                                    142
Senior Business Analyst                                     138
IT Business Analyst                                          53
Business System Analyst                                      53
Business Analyst II                                          49
Tax-Partnership Allocation Reporting Solutions - Manager     43
Technical Business Analyst                                   42
Senior Business Systems Analyst                              38
Store Manager                                                36
Business Analyst III                                       

In [11]:
# Step 11: Build initial role_category keyword classifier
def assign_role_category(title):
    t = title
    has_ba_abbrev = bool(re.search(r'\bba\b', t))
    
    if 'business analyst' in t or 'business system analyst' in t or 'business process analyst' in t or has_ba_abbrev:
        return 'Business Analysis'
    elif 'data analyst' in t or 'analytics' in t or re.search(r'\bdata\b', t):
        return 'Data & Analytics'
    elif 'project manager' in t or 'program manager' in t:
        return 'Project / Program Management'
    elif 'systems analyst' in t or 'system analyst' in t or 'enterprise architect' in t or 'software engineer' in t or re.search(r'\bit\b', t):
        return 'IT / Systems'
    elif 'tax' in t or 'financial analyst' in t or 'accounting' in t or 'controller' in t:
        return 'Finance / Tax'
    elif 'store' in t:
        return 'Retail / Store'
    elif 'consultant' in t:
        return 'Consulting'
    elif 'manager' in t:
        return 'Manager (other)'
    elif re.search(r'\banalyst\b', t):
        return 'Analyst (other)'
    else:
        return 'Other / Uncategorized'

combined2['role_category'] = combined2['job_title_normalized'].apply(assign_role_category)

print("Role category breakdown:")
print(combined2['role_category'].value_counts())
print(f"\nTotal: {combined2['role_category'].value_counts().sum()} (should equal {len(combined2)})")

print("\nAs % of total:")
print((100 * combined2['role_category'].value_counts() / len(combined2)).round(1))

Role category breakdown:
role_category
Business Analysis               4689
Other / Uncategorized            995
Manager (other)                  904
Analyst (other)                  575
Finance / Tax                    346
Data & Analytics                 296
Project / Program Management     279
IT / Systems                     230
Retail / Store                   153
Consulting                       143
Name: count, dtype: int64

Total: 8610 (should equal 8610)

As % of total:
role_category
Business Analysis               54.5
Other / Uncategorized           11.6
Manager (other)                 10.5
Analyst (other)                  6.7
Finance / Tax                    4.0
Data & Analytics                 3.4
Project / Program Management     3.2
IT / Systems                     2.7
Retail / Store                   1.8
Consulting                       1.7
Name: count, dtype: float64


In [12]:
# Step 12: Inspect "Other / Uncategorized" sample for missed patterns
uncategorized2 = combined2[combined2['role_category'] == 'Other / Uncategorized']
print(f"Sample of 20 titles from 'Other / Uncategorized' ({len(uncategorized2)} total):")
print(uncategorized2['job_title'].sample(20, random_state=42).tolist())

Sample of 20 titles from 'Other / Uncategorized' (995 total):
['Senior Business Banker', 'Technical Writer', 'Human Resources Business Partner', 'Senior Financial/Operational Auditor (Hybrid)', 'Underwriting - Workers Compensation', 'Administrative Specialist', 'Buyer', 'Regenerative Surgical Specialist', 'Vice President - Machine Learning COE', 'INFORMATION TECHNOLOGY SPECIALIST II', 'District Sales Mgr - Agriculture Equipment (Mid- West)', 'Principal Engineer - Dispensing Technologies', 'Technical Architect', 'Senior Principal Product Owner', 'AVP, Cyber E&O Underwriter', 'Project Engineer - 23-108072', 'Business Intelligence Lead Developer', 'FX - Business Lead', 'Dynamics CRM/CE Presales Architect - Public Sector', 'Technical Writer']


In [13]:
# Step 13: Check frequency of "Mgr" abbreviation
mgr_count = combined2['job_title_normalized'].str.contains(r'\bmgr\b', regex=True).sum()
print(f"Titles containing standalone 'mgr': {mgr_count}")

Titles containing standalone 'mgr': 14


In [14]:
# Step 14: Add "Mgr" fix to role_category classifier
def assign_role_category(title):
    t = title
    has_ba_abbrev = bool(re.search(r'\bba\b', t))
    has_mgr_abbrev = bool(re.search(r'\bmgr\b', t))
    
    if 'business analyst' in t or 'business system analyst' in t or 'business process analyst' in t or has_ba_abbrev:
        return 'Business Analysis'
    elif 'data analyst' in t or 'analytics' in t or re.search(r'\bdata\b', t):
        return 'Data & Analytics'
    elif 'project manager' in t or 'program manager' in t:
        return 'Project / Program Management'
    elif 'systems analyst' in t or 'system analyst' in t or 'enterprise architect' in t or 'software engineer' in t or re.search(r'\bit\b', t):
        return 'IT / Systems'
    elif 'tax' in t or 'financial analyst' in t or 'accounting' in t or 'controller' in t:
        return 'Finance / Tax'
    elif 'store' in t:
        return 'Retail / Store'
    elif 'consultant' in t:
        return 'Consulting'
    elif 'manager' in t or has_mgr_abbrev:
        return 'Manager (other)'
    elif re.search(r'\banalyst\b', t):
        return 'Analyst (other)'
    else:
        return 'Other / Uncategorized'

combined2['role_category'] = combined2['job_title_normalized'].apply(assign_role_category)

print("Role category breakdown (final, after Mgr fix):")
print(combined2['role_category'].value_counts())
print(f"\nTotal: {combined2['role_category'].value_counts().sum()} (should equal {len(combined2)})")

print("\nAs % of total:")
print((100 * combined2['role_category'].value_counts() / len(combined2)).round(1))

Role category breakdown (final, after Mgr fix):
role_category
Business Analysis               4689
Other / Uncategorized            981
Manager (other)                  918
Analyst (other)                  575
Finance / Tax                    346
Data & Analytics                 296
Project / Program Management     279
IT / Systems                     230
Retail / Store                   153
Consulting                       143
Name: count, dtype: int64

Total: 8610 (should equal 8610)

As % of total:
role_category
Business Analysis               54.5
Other / Uncategorized           11.4
Manager (other)                 10.7
Analyst (other)                  6.7
Finance / Tax                    4.0
Data & Analytics                 3.4
Project / Program Management     3.2
IT / Systems                     2.7
Retail / Store                   1.8
Consulting                       1.7
Name: count, dtype: float64


In [15]:
# Step 15: Inspect "Other / Uncategorized" by frequency for further patterns
uncategorized2 = combined2[combined2['role_category'] == 'Other / Uncategorized']

print(f"Top 40 most common titles in Other/Uncategorized ({len(uncategorized2)} total):")
print(uncategorized2['job_title_normalized'].value_counts().head(40))

Top 40 most common titles in Other/Uncategorized (981 total):
job_title_normalized
account executive digital sales                                         10
customer integrations team lead                                          9
senior accountant                                                        8
senior system engineer                                                   6
supply chain solutions delivery lead -                                   5
solutions architect                                                      5
account executive                                                        5
scrum master                                                             5
solution architect                                                       4
accountant                                                               4
servicenow architect                                                     4
servicenow lead                                                          4
sales capture exe

In [16]:
# Step 16: Quantify candidate patterns
patterns_to_check = {
    'engineer/architect/developer': r'\b(engineer|architect|developer)\b',
    'accountant/auditor/underwriter': r'\b(accountant|auditor|underwriter)\b',
    'human resources / hr': r'human resources|\bhr\b',
    'sales / account executive': r'\bsales\b|account executive',
    'scrum master/agile lead': r'scrum master',
}

for label, pattern in patterns_to_check.items():
    count = uncategorized2['job_title_normalized'].str.contains(pattern, regex=True).sum()
    print(f"{label}: {count}")

engineer/architect/developer: 274
accountant/auditor/underwriter: 82
human resources / hr: 16
sales / account executive: 81
scrum master/agile lead: 8


/tmp/ipykernel_58/2713716408.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  count = uncategorized2['job_title_normalized'].str.contains(pattern, regex=True).sum()


In [17]:
# Step 17: Finalize role_category classifier
def assign_role_category(title):
    t = title
    has_ba_abbrev = bool(re.search(r'\bba\b', t))
    has_mgr_abbrev = bool(re.search(r'\bmgr\b', t))
    
    if 'business analyst' in t or 'business system analyst' in t or 'business process analyst' in t or has_ba_abbrev:
        return 'Business Analysis'
    elif 'data analyst' in t or 'analytics' in t or re.search(r'\bdata\b', t):
        return 'Data & Analytics'
    elif 'project manager' in t or 'program manager' in t:
        return 'Project / Program Management'
    elif 'systems analyst' in t or 'system analyst' in t or 'enterprise architect' in t or 'software engineer' in t or re.search(r'\bit\b', t):
        return 'IT / Systems'
    elif re.search(r'\b(engineer|architect|developer)\b', t):
        return 'Engineering / Architecture (other)'
    elif 'tax' in t or 'financial analyst' in t or 'accounting' in t or 'accountant' in t or 'auditor' in t or 'underwriter' in t or 'controller' in t:
        return 'Finance / Accounting / Tax'
    elif 'sales' in t or 'account executive' in t:
        return 'Sales / Business Development'
    elif 'store' in t:
        return 'Retail / Store'
    elif 'consultant' in t:
        return 'Consulting'
    elif 'manager' in t or has_mgr_abbrev:
        return 'Manager (other)'
    elif re.search(r'\banalyst\b', t):
        return 'Analyst (other)'
    else:
        return 'Other / Uncategorized'

combined2['role_category'] = combined2['job_title_normalized'].apply(assign_role_category)

print("Role category breakdown (final, 13 categories):")
print(combined2['role_category'].value_counts())
print(f"\nTotal: {combined2['role_category'].value_counts().sum()} (should equal {len(combined2)})")

print("\nAs % of total:")
print((100 * combined2['role_category'].value_counts() / len(combined2)).round(1))

Role category breakdown (final, 13 categories):
role_category
Business Analysis                     4689
Manager (other)                        872
Analyst (other)                        557
Other / Uncategorized                  540
Finance / Accounting / Tax             429
Data & Analytics                       296
Engineering / Architecture (other)     283
Project / Program Management           279
IT / Systems                           230
Retail / Store                         151
Sales / Business Development           145
Consulting                             139
Name: count, dtype: int64

Total: 8610 (should equal 8610)

As % of total:
role_category
Business Analysis                     54.5
Manager (other)                       10.1
Analyst (other)                        6.5
Other / Uncategorized                  6.3
Finance / Accounting / Tax             5.0
Data & Analytics                       3.4
Engineering / Architecture (other)     3.3
Project / Program Management    

In [18]:
# Step 18: Explode job_skills into one row per skill, remove empty rows from missing-skill postings
skills_exploded3 = combined2.copy()
skills_exploded3['job_skills'] = skills_exploded3['job_skills'].fillna('')
skills_exploded3['job_skills'] = skills_exploded3['job_skills'].str.split(',')
skills_exploded3 = skills_exploded3.explode('job_skills')
skills_exploded3['job_skills'] = skills_exploded3['job_skills'].str.strip()

# Remove empty-string rows created by postings with no skills at all
before_drop = len(skills_exploded3)
skills_exploded3 = skills_exploded3[skills_exploded3['job_skills'] != '']
after_drop = len(skills_exploded3)

print(f"Postings: {len(combined2)}")
print(f"Rows after exploding (before removing empties): {before_drop}")
print(f"Rows after removing empty-string skills: {after_drop}")
print(f"Empty-skill rows removed: {before_drop - after_drop}")

# Case-fold
skills_exploded3['job_skills_normalized'] = skills_exploded3['job_skills'].str.lower()

distinct_before = skills_exploded3['job_skills'].nunique()
distinct_after = skills_exploded3['job_skills_normalized'].nunique()
print(f"\nDistinct skill strings before case-fold: {distinct_before}")
print(f"Distinct skill strings after case-fold: {distinct_after}")

Postings: 8610
Rows after exploding (before removing empties): 226393
Rows after removing empty-string skills: 226269
Empty-skill rows removed: 124

Distinct skill strings before case-fold: 54768
Distinct skill strings after case-fold: 47652


In [19]:
# Step 19: Case-fold skill text and check distinct skill counts
anchor_skills = {'business analysis', 'business analyst'}
co_occurring3 = skills_exploded3[~skills_exploded3['job_skills_normalized'].isin(anchor_skills)]

# Deduplicate per posting before counting to avoid double-counting
skill_posting_counts = co_occurring3.drop_duplicates(subset=['job_link', 'job_skills_normalized'])['job_skills_normalized'].value_counts()

total_postings = len(combined2)
skill_pct = (100 * skill_posting_counts / total_postings).round(2)

print("Top 20 complementary skills (% of postings):")
print(skill_pct.head(20))

Top 20 complementary skills (% of postings):
job_skills_normalized
project management        38.30
communication             35.48
data analysis             30.51
problem solving           22.10
sql                       21.52
analytical skills         19.47
communication skills      16.16
teamwork                  16.07
leadership                15.77
bachelor's degree         13.04
requirements gathering    12.61
problemsolving            11.99
agile                     11.77
collaboration             11.71
attention to detail       10.42
excel                      9.86
time management            9.58
microsoft office suite     9.33
data visualization         9.21
reporting                  9.01
Name: count, dtype: float64


In [20]:
# Step 20: Calculate complementary skill percentages
synonym_map = {
    'problemsolving': 'problem solving',
    'problemsolving skills': 'problem solving',
    'communication skills': 'communication',
    'written communication': 'communication',
    'verbal communication': 'communication',
    'microsoft office suite': 'microsoft office',
    'agile development': 'agile',
    'analytical thinking': 'analytical skills',
    'team leadership': 'leadership',
    'stakeholder engagement': 'stakeholder management',
}

skills_exploded3['job_skills_normalized'] = skills_exploded3['job_skills_normalized'].replace(synonym_map)

co_occurring3 = skills_exploded3[~skills_exploded3['job_skills_normalized'].isin(anchor_skills)]
skill_posting_counts = co_occurring3.drop_duplicates(subset=['job_link', 'job_skills_normalized'])['job_skills_normalized'].value_counts()
skill_pct = (100 * skill_posting_counts / total_postings).round(2)

print("Top 20 complementary skills (% of postings) after normalization:")
print(skill_pct.head(20))

Top 20 complementary skills (% of postings) after normalization:
job_skills_normalized
communication             55.02
problem solving           38.93
project management        38.30
data analysis             30.51
analytical skills         22.92
sql                       21.52
leadership                19.14
teamwork                  16.07
agile                     15.42
microsoft office          14.05
bachelor's degree         13.04
requirements gathering    12.61
collaboration             11.71
attention to detail       10.42
excel                      9.86
stakeholder management     9.64
time management            9.58
data visualization         9.21
reporting                  9.01
risk management            8.80
Name: count, dtype: float64


In [21]:
# Step 21: Parse job_location and inspect distinct location formats
combined2['location_last_part'] = combined2['job_location'].str.split(',').str[-1].str.strip()

print(f"Distinct raw job_location values: {combined2['job_location'].nunique()}")
print(f"(Document reports 1,564)")

print(f"\nDistinct right-side values: {combined2['location_last_part'].nunique()}")
print(combined2['location_last_part'].value_counts().head(20))

Distinct raw job_location values: 1564
(Document reports 1,564)

Distinct right-side values: 118
location_last_part
CA               869
TX               667
FL               561
NY               456
VA               438
United States    395
IL               372
OH               347
NJ               339
NC               320
MA               299
GA               294
PA               288
MI               224
WA               215
MD               168
WI               167
CO               149
MO               148
MN               148
Name: count, dtype: int64


In [22]:
# Step 22: Build state-extraction logic (code, name, country, metro lookup) and apply
us_state_codes = {
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN','IA','KS','KY','LA',
    'ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK',
    'OR','PA','RI','SC','SD','TN','TX','UT','VT','VA','WA','WV','WI','WY','DC'
}

state_name_to_code = {
    'alabama':'AL','alaska':'AK','arizona':'AZ','arkansas':'AR','california':'CA','colorado':'CO',
    'connecticut':'CT','delaware':'DE','florida':'FL','georgia':'GA','hawaii':'HI','idaho':'ID',
    'illinois':'IL','indiana':'IN','iowa':'IA','kansas':'KS','kentucky':'KY','louisiana':'LA',
    'maine':'ME','maryland':'MD','massachusetts':'MA','michigan':'MI','minnesota':'MN',
    'mississippi':'MS','missouri':'MO','montana':'MT','nebraska':'NE','nevada':'NV',
    'new hampshire':'NH','new jersey':'NJ','new mexico':'NM','new york':'NY','north carolina':'NC',
    'north dakota':'ND','ohio':'OH','oklahoma':'OK','oregon':'OR','pennsylvania':'PA',
    'rhode island':'RI','south carolina':'SC','south dakota':'SD','tennessee':'TN','texas':'TX',
    'utah':'UT','vermont':'VT','virginia':'VA','washington':'WA','west virginia':'WV',
    'wisconsin':'WI','wyoming':'WY'
}

non_us_countries = {'canada', 'mexico', 'turkey'}

metro_to_state = {
    'san francisco bay area': 'CA', 'greater houston': 'TX', 'atlanta metropolitan area': 'GA',
    'greater minneapolis-st. paul area': 'MN', 'greater milwaukee': 'WI', 'greater cleveland': 'OH',
    'dallas-fort worth metroplex': 'TX', 'greater philadelphia': 'PA', 'miami-fort lauderdale area': 'FL',
    'nashville metropolitan area': 'TN', 'killeen-temple area': 'TX', 'greater tuscaloosa area': 'AL',
    'louisville metropolitan area': 'KY', 'topeka metropolitan area': 'KS', 'greater pittsburgh region': 'PA',
    'memphis metropolitan area': 'TN', 'buffalo-niagara falls area': 'NY', 'greater richmond region': 'VA',
    'greater sacramento': 'CA', 'greater rockford area': 'IL', 'charlotte metro': 'NC',
    'cincinnati metropolitan area': 'OH', 'greater st. louis': 'MO', 'knoxville metropolitan area': 'TN',
    'greater madison area': 'WI', 'greater wilmington area': 'DE', 'baton rouge metropolitan area': 'LA',
    'los angeles metropolitan area': 'CA', 'brownsville metropolitan area': 'TX', 'greater chicago area': 'IL',
    'erie-meadville area': 'PA', 'des moines metropolitan area': 'IA', 'bellingham metropolitan area': 'WA',
    'greater savannah area': 'GA', 'greater boston': 'MA', 'greater jacksonville, nc area': 'NC',
    'pueblo-cañon city area': 'CO',
}

def extract_state(location, last_part):
    last_stripped = last_part.strip()
    loc_lower = location.lower()
    
    if last_stripped in us_state_codes:
        return last_stripped
    for name, code in state_name_to_code.items():
        if name in loc_lower:
            return code
    if 'district of columbia' in loc_lower:
        return 'DC'
    for country in non_us_countries:
        if re.search(r'\b' + country + r'\b', loc_lower):
            return 'NON_US'
    for name, code in metro_to_state.items():
        if name in loc_lower:
            return code
    return None

combined2['state'] = combined2.apply(lambda row: extract_state(row['job_location'], row['location_last_part']), axis=1)

resolved_us = ((combined2['state'].notna()) & (combined2['state'] != 'NON_US')).sum()
non_us = (combined2['state'] == 'NON_US').sum()
unresolved = combined2['state'].isna().sum()

print(f"Resolved to a US state: {resolved_us}")
print(f"Identified as non-US: {non_us}")
print(f"Still unresolved: {unresolved}")
print(f"Total: {len(combined2)}")

print(f"\nRemaining unresolved values:")
print(combined2.loc[combined2['state'].isna(), 'job_location'].value_counts().head(20).to_string())

Resolved to a US state: 8532
Identified as non-US: 20
Still unresolved: 58
Total: 8610

Remaining unresolved values:
job_location
United States                       41
Utica-Rome Area                      5
Greater Phoenix Area                 2
Greater Reno Area                    2
Greater Syracuse-Auburn Area         2
Greater Orlando                      1
Greater Macon                        1
Greater Anchorage Area               1
Tallahassee Metropolitan Area        1
Salt Lake City Metropolitan Area     1
Greater Tampa Bay Area               1


In [23]:
# Step 23: Extend metro-area lookup with newly found patterns and reapply
metro_to_state_additions = {
    'utica-rome area': 'NY',
    'greater phoenix area': 'AZ',
    'greater reno area': 'NV',
    'greater syracuse-auburn area': 'NY',
    'greater orlando': 'FL',
    'greater macon': 'GA',
    'greater anchorage area': 'AK',
    'tallahassee metropolitan area': 'FL',
    'salt lake city metropolitan area': 'UT',
    'greater tampa bay area': 'FL',
}
metro_to_state.update(metro_to_state_additions)

combined2['state'] = combined2.apply(lambda row: extract_state(row['job_location'], row['location_last_part']), axis=1)

resolved_us = ((combined2['state'].notna()) & (combined2['state'] != 'NON_US')).sum()
non_us = (combined2['state'] == 'NON_US').sum()
unresolved = combined2['state'].isna().sum()

print(f"Resolved to a US state: {resolved_us}")
print(f"Identified as non-US: {non_us}")
print(f"Still unresolved: {unresolved}")

print(f"\nRemaining unresolved values:")
print(combined2.loc[combined2['state'].isna(), 'job_location'].value_counts().to_string())

Resolved to a US state: 8549
Identified as non-US: 20
Still unresolved: 41

Remaining unresolved values:
job_location
United States    41


In [24]:
# Step 24: Top 15 states
state_counts2 = combined2[combined2['state'].notna() & (combined2['state'] != 'NON_US')]['state'].value_counts()

print(f"Postings resolved to a state: {state_counts2.sum()}")
print(f"Distinct states represented: {len(state_counts2)}")
print(f"\nTop 15 states by postings requiring Business Analysis capability:")
print(state_counts2.head(15))

print(f"\nAs % of all 8,610 postings:")
print((100 * state_counts2.head(15) / len(combined2)).round(1))

Postings resolved to a state: 8549
Distinct states represented: 51

Top 15 states by postings requiring Business Analysis capability:
state
CA    904
TX    705
FL    581
NY    520
VA    444
IL    386
OH    366
NJ    364
NC    332
GA    320
MA    311
PA    310
MI    235
WA    222
MD    180
Name: count, dtype: int64

As % of all 8,610 postings:
state
CA    10.5
TX     8.2
FL     6.7
NY     6.0
VA     5.2
IL     4.5
OH     4.3
NJ     4.2
NC     3.9
GA     3.7
MA     3.6
PA     3.6
MI     2.7
WA     2.6
MD     2.1
Name: count, dtype: float64


In [25]:
# Step 25: Save final postings and skills export files
final_export2 = combined2[[
    'job_link', 'job_title', 'job_title_normalized', 'role_category',
    'company', 'job_location', 'state', 'first_seen', 'job_level', 'job_type',
    'is_likely_repost'
]].copy()

final_export2.to_csv('/kaggle/working/ba_refined_postings_final.csv', index=False)
print(f"Saved postings table: {final_export2.shape}")

skills_final_export2 = skills_exploded3[['job_link', 'job_skills', 'job_skills_normalized']].copy()
skills_final_export2.to_csv('/kaggle/working/ba_refined_skills_final.csv', index=False)
print(f"Saved skills table: {skills_final_export2.shape}")

Saved postings table: (8610, 11)
Saved skills table: (226269, 3)


In [26]:
completeness_vars = ['job_link', 'job_title', 'role_category', 'job_skills', 'job_location', 'state', 'job_level']

summary_rows = []
total = len(combined2)

for col in completeness_vars:
    if col == 'state':
        # A state is only "populated" if it's a real resolved state — 
        # exclude both genuine nulls AND the "NON_US" placeholder
        populated = ((combined2[col].notna()) & (combined2[col] != 'NON_US')).sum()
    else:
        populated = combined2[col].notna().sum()
    
    null = total - populated
    complete_pct = 100 * populated / total
    summary_rows.append({
        'Variable': col,
        'Populated': populated,
        'Null': null,
        'Complete %': f"{complete_pct:.2f}%"
    })

completeness_summary = pd.DataFrame(summary_rows)
print(f"Completeness summary ({total}-record population):")
print(completeness_summary.to_string(index=False))

Completeness summary (8610-record population):
     Variable  Populated  Null Complete %
     job_link       8610     0    100.00%
    job_title       8610     0    100.00%
role_category       8610     0    100.00%
   job_skills       8486   124     98.56%
 job_location       8610     0    100.00%
        state       8549    61     99.29%
    job_level       8610     0    100.00%
